In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt



In [ ]:
# --- 🛠️ 配置区域 (请修改这里) ---
# 指定你要测试的 CSV 文件路径
TEST_FILE_PATH = "./TraningData/5/X_65.csv"
# TEST_FILE_PATH = "./TraningData/4/O_55.csv"

# 模型路径
MODEL_PATH = "./Models/gesture_model.h5"

# 类别定义 (对应 0, 1, 2, 3)
# 请根据你实际采集的顺序修改这里
CLASS_LABELS = {
    0: "Left (左)",
    1: "Right (右)",
    2: "Up (上)",
    3: "Down (下)",
    4: "O",
    5: "X",
    6: "D",
    7: "IDEL (空闲)"
}

# 数据参数
TIME_STEPS = 128
CHANNELS = 3

In [ ]:
def load_and_process_single_file(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"找不到文件: {file_path}")

    try:
        # 1. 读取并清洗 (同训练逻辑)
        df = pd.read_csv(file_path, header=None, names=['x', 'y', 'z'], on_bad_lines='skip')

        # 强制转数字，非数字变成 NaN
        df['x'] = pd.to_numeric(df['x'], errors='coerce')
        df['y'] = pd.to_numeric(df['y'], errors='coerce')
        df['z'] = pd.to_numeric(df['z'], errors='coerce')

        # 删除脏数据行
        df.dropna(inplace=True)

        data = df.values.astype(np.float32)

        if len(data) == 0:
            raise ValueError("文件为空或全是脏数据")

        # 2. 长度标准化 (补零或截断)
        if data.shape[0] < TIME_STEPS:
            padding = np.zeros((TIME_STEPS - data.shape[0], CHANNELS), dtype=np.float32)
            data = np.vstack((data, padding))
        elif data.shape[0] > TIME_STEPS:
            data = data[:TIME_STEPS, :]

        # 3. 增加 Batch 维度: (128, 3) -> (1, 128, 3)
        # 因为模型预测时需要一次输入一批数据
        data_batch = np.expand_dims(data, axis=0)

        return data, data_batch

    except Exception as e:
        print(f"处理失败: {e}")
        return None, None

In [ ]:
# 1. 加载数据
original_data, input_tensor = load_and_process_single_file(TEST_FILE_PATH)

if input_tensor is not None:
    # 2. 加载模型
    if not os.path.exists(MODEL_PATH):
        print(f"❌ 错误: 找不到模型文件 {MODEL_PATH}，请先运行训练脚本。")
    else:
        print(f"正在加载模型: {MODEL_PATH} ...")
        model = tf.keras.models.load_model(MODEL_PATH)

        # 3. 预测
        print("正在推理...")
        predictions = model.predict(input_tensor)

        # 4. 解析结果
        predicted_class_idx = np.argmax(predictions[0]) # 概率最大的下标
        predicted_prob = np.max(predictions[0])         # 最大概率值

        result_name = CLASS_LABELS.get(predicted_class_idx, "Unknown")

        # --- 打印结果 ---
        print("-" * 30)
        print(f"📂 测试文件: {TEST_FILE_PATH}")
        print(f"🎯 预测结果: [ ID: {predicted_class_idx} ] -> {result_name}")
        print(f"📊 置信度:   {predicted_prob:.2%}")
        print("-" * 30)
        print("各类别概率分布:")
        for idx, prob in enumerate(predictions[0]):
            label = CLASS_LABELS.get(idx, str(idx))
            print(f"  - {label}: {prob:.4f}")

In [ ]:
if original_data is not None:
    plt.figure(figsize=(12, 4))
    plt.plot(original_data)
    plt.title(f"Waveform: {os.path.basename(TEST_FILE_PATH)}")
    plt.xlabel("Time Steps")
    plt.ylabel("Accel Value")
    plt.legend(['X', 'Y', 'Z'], loc='upper right')
    plt.grid(True)
    plt.show()